---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: d:\SOCASIS\Ingineria AI\echochamber-project-team-4
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [3]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[RecorderRomania] Păi de ce nu au sunat la poliție să reclame faptul ca se face o ilegalitate. Băi
[b1tvchannel] Despre Georgescu, nu cred că habaucii, analfabeții care, de fiecare dată când es
[georgesimionoficial] Si inca o chestie, de 5 ani si ceva, Simion e in parlamentul Romaniei. Ce a facu


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [4]:
# modifica dupa preferinte
AXA_1 = "anti_system"
AXA_2 = "emotional_intensity"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [5]:
# Celula de cod pentru definiții
AXA_1_DEFINITION = """
anti_system măsoară măsura în care comentariul atacă, respinge sau critică instituțiile statului, politicienii (clasa politică în general) sau "sistemul".
0 = absent (niciun atac la adresa sistemului)
1 = prezent (critică moderată sau o mențiune negativă)
2 = dominant (comentariul este un atac vehement, concentrat exclusiv pe dărâmarea sistemului)
"""

AXA_2_DEFINITION = """
emotional_intensity măsoară încărcătura emoțională și agresivitatea limbajului folosit.
0 = neutru (limbaj calm, faptic, fără emoții vizibile)
1 = emoțional (exprimă supărare, speranță, frustrare, dar păstrează o decență)
2 = agresiv/extrem (folosește insulte, ură, semne de exclamare multiple, majuscule, limbaj extrem)
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [6]:
SYSTEM_PROMPT = """Ești un asistent de cercetare expert în științe politice și analiza discursului pe platformele sociale. 
Sarcina ta este să adnotezi comentarii YouTube pe două axe specifice. Ești extrem de obiectiv și respecți strict definițiile primite."""

USER_PROMPT_TEMPLATE = """
Analizează următorul comentariu politic și evaluează-l pe cele două axe de mai jos:

1. anti_system: {axa_1_def}
2. emotional_intensity: {axa_2_def}

Reguli de codare:
- Nu inventa atitudini care nu există clar în text.
- Evaluează doar textul pus la dispoziție.

Returnează DOAR un obiect JSON valid, fără formatare Markdown suplimentară (fără ```json), cu exact aceste 3 chei:
"anti_system" (valoare numerică: 0, 1 sau 2),
"emotional_intensity" (valoare numerică: 0, 1 sau 2),
"explicatie" (o frază scurtă care justifică scorurile).

Comentariu de analizat:
<<< {comment_text} >>>
"""

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [8]:
TESTS = corpus.sample(5, random_state=42)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
145,yt_BfvZ8QcVBKc_Ugyog0iMEqAX5zIQb4R4AaABAg,turcescu111,Harpalete- Sângerete și transfuzia din lumea lui,"Da, si eu cred ca serviciile ucrainene au fost..."
334,yt_qkGhsJFft00_UgzJHKCoGkwtTJ3uOBh4AaABAg,digi24hd56,În fața ta cu Emil Hurezeanu: „Ar fi un coșmar...,Ce mă supără pe mine oamenii ăștia care sunt a...
175,yt_KqrUotq1Obs_Ugy6tYmdfH0T2THArnB4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pace și prosperitate ( 28.10...,Bunul Dumnezeu să îl protejeze pe președintele...
369,yt_vkP6FdP9iX0_Ugwj3HojTt6JikJVtjd4AaABAg,turcescu111,Orientul Mijlociu în flăcări,Nicușor merge pe lângă covor pentru că nu are ...
416,yt_Sj4fQKlMOro_UgyNWIwNDtUeTTCrLkl4AaABAg,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Trebuie susținută aceasta femeie!!!!! Acesta a...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [9]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [10]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [13]:
results = []
for _, row in TESTS.iterrows():
    USER = USER_PROMPT_TEMPLATE.format(
        axa_1_def=AXA_1_DEFINITION,
        axa_2_def=AXA_2_DEFINITION,
        comment_text=row["text"]
    )
    
    raw = llm(SYSTEM_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Da, si eu cred ca serviciile ucrainene au fost inplicate in alegerile noastre. Am convingerea ca serviciile noastre sunt infiltrate de serviciile ucrainene

OUTPUT MODEL:
```json
{
  "anti_system": 1,
  "emotional_intensity": 1,
  "explicatie": "Comentariul sugerează o implicare a serviciilor ucrainene în alegerile interne și o infiltrare a serviciilor românești, ceea ce reprezintă o critică moderată la adresa sistemului de securitate național."
}
```
COMENTARIU:
Ce mă supără pe mine oamenii ăștia care sunt așa de siguri când spun ca Iranul nu mai are arsenal militar. De unde mama dracului știu ei treburile astea. Noi bănuim ca primesc arme din Rusia bolshevica și China comunistă. Poate și Brazilia dar iarăși e o bănuială.

OUTPUT MODEL:
```json
{
  "anti_system": 1,
  "emotional_intensity": 1,
  "explicatie": "Comentariul critică siguranța cu care anumite persoane afirmă informații despre arsenalul militar al Iranului, sugerând o lipsă de transparență sau cunoaștere din pa

In [ ]:
## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?

## Interpretare

#### `anti_system` (scala 0–2) și `emotional_intensity` (scala 0–2).

#### Sunt două axe relevante pentru discursul politic românesc de pe YouTube. `anti_system` surprinde conținutul ideologic al comentariului, iar `emotional_intensity` surprinde modul de exprimare. Împreună oferă o imagine mai completă decât oricare dintre ele singură: un comentariu poate fi anti-sistem dar calm (ex. comentariul 1), sau emoțional fără atacuri la sistem (ex. comentariul 3).

#### Parțial, conținutul JSON este corect structurat și conține exact cele 3 chei cerute (`anti_system`, `emotional_intensity`, `explicatie`), cu valori numerice valide. Problema este că modelul a returnat JSON învelit în backtick-uri Markdown, deși promptul cerea explicit fără formatare Markdown. Un `json.loads()` direct pe output ar fi eșuat, este necesar un pas de curățare a string-ului înainte de parsare.

#### Promptul a funcționat bine. Modelul a respectat cu strictețe formatul JSON și a evaluat foarte corect nuanțele. A reușit să facă diferența clară între o critică moderată și o revoltă extremă anti-sistem.